In [1]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Apr 1, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA_V2
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    compute_batch_size,
    detect_device,
    load_split_bundle,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Mounted at /content/drive


## Load data

In [2]:
DATA_SET = 'rand_B'

df_train = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_train_v2.parquet'))
df_val   = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_val_v2.parquet'))
df_test  = pd.read_parquet(CLEAN_DATA_V2 / (DATA_SET + '_test_v2.parquet'))


OUTPUT_DIR = make_run_dir()

## GPU auto-detect

In [3]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

L4  |  VRAM: 24 GB  |  MAX_BATCH=32,768  |  dtype=torch.bfloat16


## Hyperparameters

In [4]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BATCH_SIZE = compute_batch_size(len(df_train), CFG['MAX_BATCH'])
BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

steps_per_epoch = len(df_train) // BATCH_SIZE
print(f'MAX_BATCH={CFG["MAX_BATCH"]:,}  adaptive BATCH_SIZE={BATCH_SIZE:,}  '
      f'INIT_LR={INIT_LR:.6f}  n_train={len(df_train):,}  '
      f'steps/epoch~{steps_per_epoch}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'WARMUP={WARMUP_EPOCHS} epochs')

MAX_BATCH=32,768  adaptive BATCH_SIZE=16,384  INIT_LR=0.002000  n_train=911,172  steps/epoch~55
MAX_EPOCHS=100  PATIENCE=30  WARMUP=5 epochs


## Feature definitions

In [5]:
from itertools import combinations

FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

# Generate ALL possible combinations of EXTRA_FEATURES (0 to 7)
feature_combos = []
base_name = '+'.join(FEATURES_3)
feature_combos.append((base_name, FEATURES_3.copy()))  # Base case

for r in range(1, len(EXTRA_FEATURES) + 1):
    for combo in combinations(EXTRA_FEATURES, r):
        feats = FEATURES_3 + list(combo)
        name = '+'.join(feats)
        feature_combos.append((name, feats))

# Calculate breakdown for all sizes
n_plus_0 = 1
n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))
n_plus_4 = len(list(combinations(EXTRA_FEATURES, 4)))
n_plus_5 = len(list(combinations(EXTRA_FEATURES, 5)))
n_plus_6 = len(list(combinations(EXTRA_FEATURES, 6)))
n_plus_7 = len(list(combinations(EXTRA_FEATURES, 7)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')
print(f'  3F + 4:     {n_plus_4}')
print(f'  3F + 5:     {n_plus_5}')
print(f'  3F + 6:     {n_plus_6}')
print(f'  3F + 7:     {n_plus_7}')

Total feature combinations: 128
  Base (3F):  1
  3F + 1:     7
  3F + 2:     21
  3F + 3:     35
  3F + 4:     35
  3F + 5:     21
  3F + 6:     7
  3F + 7:     1


## Pre-allocate on GPU

In [6]:
gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Data on GPU  |  VRAM used: 0.26 GB / 24 GB  |  Free: 23.4 GB
Train: 911,172  Val: 260,336  Test: 130,168  Features: 10


## Analytic benchmark

In [7]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 38.6267  RMSE = 0.017226
Coefficients: a = -0.092294, b = -0.154601, c = -0.144622


## Train all feature combinations

In [8]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    nan_mask_tr=gpu_data['nan_mask_tr'],
    nan_mask_va=gpu_data['nan_mask_va'],
    nan_mask_ytr=gpu_data['nan_mask_ytr'],
    nan_mask_yva=gpu_data['nan_mask_yva'],
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

sep = "=" * 70
print('\n' + sep)
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(sep + '\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 128 MODELS
  GPU: L4  |  Batch: 16,384  |  AMP: True  |  Max epochs: 100

  [  1/128] delta+T+spy_ret                SSE=38.5548  Gain=+0.2%  ep=100  19.8s  elapsed=0.3min
  [ 25/128] delta+T+spy_ret+gamma+vega     SSE=38.7986  Gain=-0.4%  ep=100  13.5s  elapsed=5.7min
  [ 50/128] delta+T+spy_ret+vix_mom_lag+gamma+vega SSE=35.8114  Gain=+7.3%  ep=100  13.6s  elapsed=11.4min
  [ 75/128] delta+T+spy_ret+vix_lag+vix_mom+gamma+theta SSE=33.4108  Gain=+13.5%  ep=100  13.8s  elapsed=17.1min
  [100/128] delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+theta SSE=29.1399  Gain=+24.6%  ep=100  13.6s  elapsed=22.7min
  [125/128] delta+T+spy_ret+vix_lag+vix_mom_lag+gamma+theta+vega+rho SSE=35.8357  Gain=+7.2%  ep=100  13.7s  elapsed=28.4min
  [128/128] delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+theta+vega+rho SSE=29.8800  Gain=+22.6%  ep=100  13.5s  elapsed=29.1min

Done: 29.1 min for 128 models (avg 13.6s/model)


## Results summary

In [9]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,38.626713,0.000297,0.017226,0.006097,0.001861,0.002207,0.067476,None,None,None
1,delta+T+spy_ret,38.554832,0.000296,0.017210,0.007769,0.000078,0.004303,0.069211,14.5s,0.19%,None
2,delta+T+spy_ret+vix_lag,46.512840,0.000357,0.018903,0.011178,-0.000204,0.007583,-0.122911,13.7s,-20.42%,-20.64%
3,delta+T+spy_ret+vix_mom_lag,46.069633,0.000354,0.018813,0.010924,-0.000475,0.007264,-0.112211,13.5s,-19.27%,0.95%
4,delta+T+spy_ret+vix_mom,47.629082,0.000366,0.019129,0.011171,0.001200,0.007507,-0.149859,13.5s,-23.31%,-3.38%
...,...,...,...,...,...,...,...,...,...,...,...
125,delta+T+spy_ret+vix_lag+vix_mom_lag+gamma+thet...,35.835728,0.000275,0.016592,0.009391,-0.000894,0.006104,0.134855,13.7s,7.23%,-5.96%
126,delta+T+spy_ret+vix_lag+vix_mom+gamma+theta+ve...,34.817398,0.000267,0.016355,0.009285,-0.001860,0.006157,0.159440,13.4s,9.86%,2.84%
127,delta+T+spy_ret+vix_mom_lag+vix_mom+gamma+thet...,35.154087,0.000270,0.016434,0.009221,-0.001083,0.006019,0.151312,13.8s,8.99%,-0.97%
128,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,29.880026,0.000230,0.015151,0.008153,0.000227,0.005163,0.278638,13.5s,22.64%,15.00%


## Top 10 by Gain vs Hull-White

In [10]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+th...,8,28.6926,0.014847,25.72,13.7
1,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,28.7859,0.014871,25.48,13.8
2,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+th...,8,29.0533,0.014940,24.78,13.4
3,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,29.1399,0.014962,24.56,13.6
4,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ve...,8,29.4429,0.015040,23.78,13.4
5,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,29.5463,0.015066,23.51,13.7
6,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,10,29.8800,0.015151,22.64,13.5
7,delta+T+spy_ret+vix_mom_lag+vix_mom+gamma+thet...,8,30.2354,0.015241,21.72,13.4
8,delta+T+spy_ret+vix_lag+vix_mom+gamma+vega,7,30.2950,0.015256,21.57,13.5
9,delta+T+spy_ret+vix_lag+vix_mom+gamma+rho,7,30.5446,0.015318,20.92,13.4


## Best per group (3F, +1, +2, +3)

In [11]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f"{nf_label}: {best['combo_name']}")
    print(f"    SSE={best['SSE']:.4f}  RMSE={best['RMSE']:.6f}  Gain={best['Gain_vs_HW_%']:.2f}%\n")

3F (base): delta+T+spy_ret
    SSE=38.5548  RMSE=0.017210  Gain=0.19%

+1 (4F): delta+T+spy_ret+theta
    SSE=43.0728  RMSE=0.018191  Gain=-11.51%

+2 (5F): delta+T+spy_ret+vix_lag+vix_mom_lag
    SSE=33.2313  RMSE=0.015978  Gain=13.97%

+3 (6F): delta+T+spy_ret+vix_lag+vix_mom_lag+vega
    SSE=32.6667  RMSE=0.015842  Gain=15.43%



## Summary statistics

In [12]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 29.0 min (0.48 hr)
Models trained: 128
Best overall: delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+theta+vega (Gain=25.72%)


## Cleanup

In [13]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.38 GB / 24 GB
Total training time: 29.0 min for 128 models
